In [1]:
import warnings
from pathlib import Path

import xarray as xr
import numpy as np

from imagematerials.buildings.preprocessing.floorspace import (
    compute_average_m2_capita,
    compute_housing_residential,
    compute_housing_type,
    extrapolate_floorspace,
    get_image_floorspace
)
from imagematerials.buildings.preprocessing.lifetimes import compute_lifetimes
from imagematerials.buildings.preprocessing.materials import (
    compute_mat_intensities_commercial,
    compute_mat_intensities_residential,
)
from imagematerials.buildings.preprocessing.population import compute_population_split
from imagematerials.concepts import create_building_graph



from matplotlib import pyplot as plt

from imagematerials.buildings.preprocessing.population import compute_population_split

from imagematerials.buildings.constants import urban_q_areas, rural_q_areas

In [2]:
base_directory = Path("..", "data", "raw")
scenario_sel = "SSP2_baseline"

database_directory = base_directory / "buildings" / "SSP2_CP"
image_directory = base_directory / "image" / scenario_sel

In [3]:
population = compute_population_split(image_directory)

C:\Coding\image-materials\imagematerials\buildings\preprocessing\population.py:204: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'Quintile' ('Quintile',) The recommendation is to set join explicitly for this case.
  combined_quintiles = xr.concat(


In [4]:

population = compute_population_split(image_directory)
population.sum("Region").sel(Area='Urban').plot(label="Total Urban")
population.sum("Region").sel(Area=urban_q_areas).sum("Area").plot(label = "Urban Quintiles")

population.sum("Region").sel(Area='Rural').plot(label="Total Rural")
population.sum("Region").sel(Area=rural_q_areas).sum("Area").plot(label = "Rural Quintiles")

plt.legend()

C:\Coding\image-materials\imagematerials\buildings\preprocessing\population.py:204: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'Quintile' ('Quintile',) The recommendation is to set join explicitly for this case.
  combined_quintiles = xr.concat(


TypeError: Plotting requires coordinates to be numeric, boolean, or dates of type numpy.datetime64, datetime.datetime, cftime.datetime or pandas.Interval. Received data of type object instead.

In [ ]:
population

In [ ]:
# Get floorspace for commercial + urban/rural
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    floorspace_image_commercial_rururb, minimum_comm = get_image_floorspace(image_directory, base_directory)
floorspace_commercial_rururb = extrapolate_floorspace(floorspace_image_commercial_rururb, minimum_comm)

In [ ]:
from imagematerials.buildings.constants import commercial_types

region_sel = "4"

floorspace_commercial_rururb.sel(Type = "Urban").sel(Region = region_sel).plot(label = "Urban " + region_sel)
for urban_q in urban_q_areas:
    floorspace_commercial_rururb.sel(Type = urban_q).sel(Region = region_sel).plot(label = "Urban " + urban_q + " " + region_sel)
plt.title("Floorspace per Capita in Urban Areas of Region " + region_sel)

floorspace_commercial_rururb.sel(Type = commercial_types).sum("Type").sel(Region = region_sel).plot(label = "Commercial " + region_sel)
plt.legend()

In [ ]:
# Average square meter per capita split by residential type [Region, Area, Type]

from imagematerials.buildings.constants import area_graph

average_m2_capita = compute_average_m2_capita(base_directory)

print('average_m2_capita:', average_m2_capita.pint.units)
print('Area coords:', list(average_m2_capita.coords['Area'].values))

In [ ]:
housing_type = compute_housing_type(database_directory)


In [ ]:
# Rural/Urban floorspace [Time, Region, Area]
floorspace_rururb = floorspace_commercial_rururb.sel(
    {"Type": ["Urban", "Rural"]+urban_q_areas+rural_q_areas}).rename({"Type": "Area"})

In [ ]:
# Floorspace m2 for residential buildings [Year, Region, Area, Type]
floorspace_residential = compute_housing_residential(population, 
                                                     average_m2_capita, 
                                                     housing_type, 
                                                     floorspace_rururb, {"test":None})

In [ ]:
floorspace_residential

In [ ]:
# Residential housing type shares [Year, Region, Area, Type]
housing_type = compute_housing_type(database_directory)
for res_type in housing_type.coords["Type"].values:
    housing_type.mean(["Region", "Area"]).sel(Type=res_type).plot(label=res_type)
plt.title("Residential housing shares per type.")
plt.legend()
plt.show()

In [ ]:
for res_type in floorspace_residential.coords["Type"].values:
    floorspace_residential.sum(["Region"]).sel(Type=res_type).plot(label=res_type)
plt.title("Residential housing million m2 per type.")
plt.legend()
plt.show()

In [ ]:
floorspace_commercial_total = floorspace_commercial * population.sel({"Area": "Total"})

In [ ]:
circular_economy_config = {"test": None}

floorspace_commercial_total = floorspace_commercial_total.drop_vars("Area")
floorspace = xr.concat((floorspace_residential, floorspace_commercial_total), dim="Type")

# Lifetime computations, see lifetimes.py

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lifetimes = compute_lifetimes(base_directory, floorspace_commercial.coords["Type"].values, circular_economy_config)

mat_intensities_comm = compute_mat_intensities_commercial(database_directory, circular_economy_config)
mat_intensities_res = compute_mat_intensities_residential(database_directory, circular_economy_config)
mat_intensities = xr.concat((mat_intensities_res, mat_intensities_comm), dim="Type")
knowledge_graph = create_building_graph()
mat_intensities = knowledge_graph.rebroadcast_xarray(
                        mat_intensities, floorspace.coords["Type"].values)

#TODO remove this quick fix
region_coords = np.sort(floorspace.coords["Region"].values.astype(int)).astype(str)

In [ ]:
import pandas as pd
from imagematerials.read_mym import read_mym_df

from imagematerials.buildings.constants import (
    END_YEAR,
    FLAG_ALPHA,
    FLAG_EXPDEC,
    GOMPERTZ_EXPDEC,
    HIST_YEAR,
    INFLATION,
    REGIONS,
    REGIONS_RANGE,
    START_YEAR,
    YEARS,
)


                                                                              

In [ ]:
historic_population = pd.read_csv(base_directory / 'buildings' / 'standard_data'
                               / 'historic_population.csv', index_col=0, header = 0)

historic_population = historic_population.loc[:1971] / 1000 # unit conversion
historic_population

historic_population = historic_population.reindex(range(historic_population.index.min(),
                                            historic_population.index.max() + 1)
                                    ).interpolate(method="linear")


In [ ]:
pop_q_xr = population.copy()
import prism

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

rural_total = population_split_xr.sum("Region").sel(Area="Rural")
urban_total = population_split_xr.sum("Region").sel(Area="Urban")
rural_q1_x5 = population_split_xr.sum("Region").sel(Area="Rural Q1") * 5

rural_total.sel(Time=slice(1880, 1960)).plot(ax=ax, label="Rural total (smoothed)", linewidth=2)
urban_total.sel(Time=slice(1880, 1960)).plot(ax=ax, label="Urban total", linewidth=2)
rural_q1_x5.sel(Time=slice(1880, 1960)).plot(ax=ax, label="5 x Rural Q1", linestyle="--")

ax.set_title("Historic window around 1925 (global sum)")
ax.set_xlabel("Time")
ax.set_ylabel("people")
ax.legend()
plt.show()

In [ ]:
floorspace_xr = (
    floorspace_urban_rural_tot
    .set_index(["Time", "Region"])[["Urban", "Rural", 
                                    "Urban Q1", "Urban Q2", "Urban Q3", "Urban Q4", "Urban Q5",
                                   "Rural Q1", "Rural Q2", "Rural Q3", "Rural Q4", "Rural Q5"]]
    .rename_axis(columns="Type")
    .stack()
    .to_xarray()
    .transpose("Time", "Region", "Type")
)
floorspace_xr

# remove region 27 (global)
floorspace_xr = floorspace_xr.sel(Region=REGIONS_RANGE)
prism.Q_(floorspace_xr, "m^2/person")


In [ ]:
# commercial floorspace
from imagematerials.buildings.preprocessing.floorspace import compute_commercial_floor_m2_cap, compute_commercial_floor_m2_cap_sum, get_gompertz, get_service_value_added

gompertz = get_gompertz(base_directory)
service_value_added = get_service_value_added(image_directory)
commercial_m2_cap_sum = compute_commercial_floor_m2_cap_sum(gompertz, service_value_added)

commercial_m2_cap_df, minimum_comm = compute_commercial_floor_m2_cap(
        gompertz, commercial_m2_cap_sum, service_value_added)

minimum_comm = prism.Q_(minimum_comm, "m^2/person")

# Convert to xarray with Area dimension for commercial building types (Office, Retail+, Hotels+, Govt+)
commercial_m2_cap = (
    commercial_m2_cap_df
    .reset_index()
    .set_index(["Time", "Region"])
    .rename_axis(columns="Type")
    .stack()
    .to_xarray()
    .transpose("Time", "Region", "Type")
)
commercial_m2_cap = commercial_m2_cap.sel(Region=REGIONS_RANGE)
prism.Q_(commercial_m2_cap, "m^2/person")

In [ ]:
# Join commercial_m2_cap and floorspace_xr along Area dimension
floorspace_all = xr.concat([floorspace_xr, commercial_m2_cap], dim="Type")
floorspace_all.sel(Type="Urban").sel(Region = 11).loc[1971:].plot(label="Urban")
floorspace_all.sel(Type="Rural").sel(Region = 11).loc[1971:].plot(label="Rural")
floorspace_all.sel(Type=['Govt+', 'Hotels+', 'Office', 'Retail+']).sum("Type").sel(Region = 11).loc[1971:].plot(label="Hotels+")

In [ ]:
floorspace_commercial_rururb = extrapolate_floorspace(floorspace_all,
                                                          minimum_comm)